In [19]:
import pandas as pd

In [20]:
#          Identificadores directos: name, document_id, email, loyalty_card_number
#             Cuasi-identificadores: age, city, occupation
# Información sensible/confidencial: annual_spend
raw = pd.read_csv("../data/raw.csv")
raw.head()

,name,document_id,email,loyalty_card_number,age,city,occupation,annual_spend
0,Camila González,1000000001,camila.gonzalez1@gmail.example,840000000001,25,Envigado,Comerciante,2580000
1,Mateo Ospina,1000000002,mateo.ospina2@yahoo.example,840000000002,36,Cali,Ingeniero de sistemas,5400000
2,Diana Duque,1000000003,diana.duque3@outlook.example,840000000003,38,Bogotá,Docente,5320000
3,Sebastián Álvarez,1000000004,sebastian.alvarez4@yahoo.example,840000000004,67,Itagüí,Abogada,4290000
4,Mariana Henao,1000000005,mariana.henao5@outlook.example,840000000005,29,Bogotá,Comerciante,3670000


In [21]:
# Información pública externa
auxiliary = pd.read_csv("../data/auxiliary.csv")
auxiliary.head()

,name,age,city,occupation
0,Laura Gómez,47,Rionegro,Arquitecta
1,Andrés Restrepo,38,Envigado,Ingeniero de sistemas
2,Camila Henao,29,Pereira,Diseñadora gráfica
3,Carlos Muñoz,61,Bello,Técnico electricista
4,Natalia Ospina,34,Manizales,Docente


In [22]:
# Anonimización ingenua
anonymized = raw.drop(columns=["name", "document_id", "email", "loyalty_card_number"])
anonymized.head()

,age,city,occupation,annual_spend
0,25,Envigado,Comerciante,2580000
1,36,Cali,Ingeniero de sistemas,5400000
2,38,Bogotá,Docente,5320000
3,67,Itagüí,Abogada,4290000
4,29,Bogotá,Comerciante,3670000


In [23]:
# La combinación de los cuasi-identificadores puede permitir la reidentificación de individuos
quasi_identifiers = ["age", "city", "occupation"]

reidentified = auxiliary.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

reidentified.head()

,name,age,city,occupation,annual_spend
0,Laura Gómez,47,Rionegro,Arquitecta,5840000
1,Andrés Restrepo,38,Envigado,Ingeniero de sistemas,5190000
2,Camila Henao,29,Pereira,Diseñadora gráfica,2460000
3,Carlos Muñoz,61,Bello,Técnico electricista,3370000
4,Natalia Ospina,34,Manizales,Docente,2710000


In [24]:
# Paso 1: eliminación de los identificadores directos
anonymized = raw.drop(columns=["name", "email"])
anonymized.head()

,document_id,loyalty_card_number,age,city,occupation,annual_spend
0,1000000001,840000000001,25,Envigado,Comerciante,2580000
1,1000000002,840000000002,36,Cali,Ingeniero de sistemas,5400000
2,1000000003,840000000003,38,Bogotá,Docente,5320000
3,1000000004,840000000004,67,Itagüí,Abogada,4290000
4,1000000005,840000000005,29,Bogotá,Comerciante,3670000


In [25]:
# Paso 2: enmascaramiento del loyalty_card_number

loyalty_card_number = anonymized["loyalty_card_number"].astype(str).str.zfill(12)

anonymized["loyalty_card_number"] = "********" + loyalty_card_number.str[-4:]

anonymized.head()

,document_id,loyalty_card_number,age,city,occupation,annual_spend
0,1000000001,********0001,25,Envigado,Comerciante,2580000
1,1000000002,********0002,36,Cali,Ingeniero de sistemas,5400000
2,1000000003,********0003,38,Bogotá,Docente,5320000
3,1000000004,********0004,67,Itagüí,Abogada,4290000
4,1000000005,********0005,29,Bogotá,Comerciante,3670000


In [26]:
# Paso 3: pseudonimización del documento de identidad

import hashlib
import hmac

secret_key = b"clave-secreta-del-programa"


def pseudonymize(document_id):
    digest = hmac.new(
        secret_key,
        str(document_id).encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()

    return f"CUST-{digest[:12].upper()}"


anonymized["customer_id"] = anonymized["document_id"].apply(pseudonymize)

anonymized = anonymized.drop(columns=["document_id"])

anonymized.head()

,loyalty_card_number,age,city,occupation,annual_spend,customer_id
0,********0001,25,Envigado,Comerciante,2580000,CUST-EFF698B261D2
1,********0002,36,Cali,Ingeniero de sistemas,5400000,CUST-E9F4EC0BD860
2,********0003,38,Bogotá,Docente,5320000,CUST-E2C7317583CA
3,********0004,67,Itagüí,Abogada,4290000,CUST-6AD4C8020C2B
4,********0005,29,Bogotá,Comerciante,3670000,CUST-92569771F46C


In [27]:
# Paso 4: anonimización de la edad

anonymized["age_group"] = pd.cut(
    anonymized["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

anonymized = anonymized.drop(columns=["age"])

anonymized.head()

,loyalty_card_number,city,occupation,annual_spend,customer_id,age_group
0,********0001,Envigado,Comerciante,2580000,CUST-EFF698B261D2,20-29
1,********0002,Cali,Ingeniero de sistemas,5400000,CUST-E9F4EC0BD860,30-39
2,********0003,Bogotá,Docente,5320000,CUST-E2C7317583CA,30-39
3,********0004,Itagüí,Abogada,4290000,CUST-6AD4C8020C2B,60-69
4,********0005,Bogotá,Comerciante,3670000,CUST-92569771F46C,20-29


In [28]:
# Verificación

auxiliary_attack = auxiliary.copy()
auxiliary_attack["age_group"] = pd.cut(
    auxiliary_attack["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

quasi_identifiers = ["age_group", "city", "occupation"]
candidates = auxiliary_attack.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

number_of_candidates = (
    candidates.groupby("name").size().reindex(auxiliary["name"], fill_value=0)
)

unique_matches = candidates[candidates["name"].map(number_of_candidates).eq(1)]

print(
    "Perfiles reidentificados de forma única:",
    number_of_candidates.eq(1).sum(),
    "de",
    len(auxiliary),
)

unique_matches.head()

Perfiles reidentificados de forma única: 9 de 20


,name,age,city,occupation,age_group,loyalty_card_number,annual_spend,customer_id
0,Laura Gómez,47,Rionegro,Arquitecta,40-49,********0392,5840000,CUST-8ADE272FFEAF
10,Javier Cárdenas,52,Bucaramanga,Médico,50-59,********0412,6420000,CUST-391F42D42771
11,Diana Correa,45,Cali,Contadora,40-49,********0079,4180000,CUST-CB0B1C18C352
12,Felipe Vélez,33,Cartagena,Comerciante,30-39,********0600,3960000,CUST-3001E46E291C
23,Isabel Díaz,64,Barranquilla,Enfermera,60-69,********0146,3520000,CUST-57D10B018171


In [29]:
# Paso 5: generalización de ciudad a departamento

city_to_department = {
    "Medellín": "Antioquia",
    "Bello": "Antioquia",
    "Envigado": "Antioquia",
    "Itagüí": "Antioquia",
    "Rionegro": "Antioquia",
    "Bogotá": "Bogotá D.C.",
    "Cali": "Valle del Cauca",
    "Barranquilla": "Atlántico",
    "Manizales": "Caldas",
    "Pereira": "Risaralda",
    "Cartagena": "Bolívar",
    "Bucaramanga": "Santander",
}

anonymized["department"] = anonymized["city"].map(city_to_department)

anonymized = anonymized.drop(columns=["city"])

anonymized.head()

,loyalty_card_number,occupation,annual_spend,customer_id,age_group,department
0,********0001,Comerciante,2580000,CUST-EFF698B261D2,20-29,Antioquia
1,********0002,Ingeniero de sistemas,5400000,CUST-E9F4EC0BD860,30-39,Valle del Cauca
2,********0003,Docente,5320000,CUST-E2C7317583CA,30-39,Bogotá D.C.
3,********0004,Abogada,4290000,CUST-6AD4C8020C2B,60-69,Antioquia
4,********0005,Comerciante,3670000,CUST-92569771F46C,20-29,Bogotá D.C.


In [30]:
# Verificación

auxiliary_attack = auxiliary.copy()

auxiliary_attack["age_group"] = pd.cut(
    auxiliary_attack["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

auxiliary_attack["department"] = auxiliary_attack["city"].map(city_to_department)

quasi_identifiers = ["age_group", "department", "occupation"]

candidates = auxiliary_attack.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

number_of_candidates = (
    candidates.groupby("name").size().reindex(auxiliary["name"], fill_value=0)
)

unique_matches = candidates[candidates["name"].map(number_of_candidates).eq(1)]

print(
    "Perfiles reidentificados de forma única:",
    number_of_candidates.eq(1).sum(),
    "de",
    len(auxiliary),
)

unique_matches.head()

Perfiles reidentificados de forma única: 7 de 20


,name,age,city,occupation,age_group,department,loyalty_card_number,annual_spend,customer_id
20,Javier Cárdenas,52,Bucaramanga,Médico,50-59,Santander,********0412,6420000,CUST-391F42D42771
21,Diana Correa,45,Cali,Contadora,40-49,Valle del Cauca,********0079,4180000,CUST-CB0B1C18C352
22,Felipe Vélez,33,Cartagena,Comerciante,30-39,Bolívar,********0600,3960000,CUST-3001E46E291C
44,Isabel Díaz,64,Barranquilla,Enfermera,60-69,Atlántico,********0146,3520000,CUST-57D10B018171
66,Lucía Londoño,58,Pereira,Docente,50-59,Risaralda,********0355,3090000,CUST-8BB7CBF5F7B5


In [31]:
# Paso 6: generalización de departamento a región

department_to_region = {
    "Antioquia": "Andina",
    "Bogotá D.C.": "Andina",
    "Caldas": "Andina",
    "Risaralda": "Andina",
    "Santander": "Andina",
    "Atlántico": "Caribe",
    "Bolívar": "Caribe",
    "Valle del Cauca": "Pacífica",
}

anonymized["region"] = anonymized["department"].map(department_to_region)

anonymized = anonymized.drop(columns=["department"])

anonymized.head()

,loyalty_card_number,occupation,annual_spend,customer_id,age_group,region
0,********0001,Comerciante,2580000,CUST-EFF698B261D2,20-29,Andina
1,********0002,Ingeniero de sistemas,5400000,CUST-E9F4EC0BD860,30-39,Pacífica
2,********0003,Docente,5320000,CUST-E2C7317583CA,30-39,Andina
3,********0004,Abogada,4290000,CUST-6AD4C8020C2B,60-69,Andina
4,********0005,Comerciante,3670000,CUST-92569771F46C,20-29,Andina


In [32]:
# Verificación

auxiliary_attack = auxiliary.copy()

auxiliary_attack["age_group"] = pd.cut(
    auxiliary_attack["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

auxiliary_attack["department"] = auxiliary_attack["city"].map(city_to_department)

auxiliary_attack["region"] = auxiliary_attack["department"].map(department_to_region)

quasi_identifiers = ["age_group", "region", "occupation"]

candidates = auxiliary_attack.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

number_of_candidates = (
    candidates.groupby("name").size().reindex(auxiliary["name"], fill_value=0)
)

unique_matches = candidates[candidates["name"].map(number_of_candidates).eq(1)]

print(
    "Perfiles reidentificados de forma única:",
    number_of_candidates.eq(1).sum(),
    "de",
    len(auxiliary),
)

unique_matches.head()

Perfiles reidentificados de forma única: 2 de 20


,name,age,city,occupation,age_group,department,region,loyalty_card_number,annual_spend,customer_id
45,Diana Correa,45,Cali,Contadora,40-49,Valle del Cauca,Pacífica,********0079,4180000,CUST-CB0B1C18C352
126,Alejandro Martínez,41,Cali,Médico,40-49,Valle del Cauca,Pacífica,********0427,5970000,CUST-935387D17037


In [33]:
# Paso 7: generalización de ocupación

occupation_to_group = {
    "Administradora": "Servicios profesionales",
    "Abogada": "Servicios profesionales",
    "Analista financiera": "Servicios profesionales",
    "Contadora": "Servicios profesionales",
    "Arquitecta": "Tecnología y diseño",
    "Diseñadora gráfica": "Tecnología y diseño",
    "Ingeniero de sistemas": "Tecnología y diseño",
    "Médico": "Salud y educación",
    "Enfermera": "Salud y educación",
    "Docente": "Salud y educación",
    "Comerciante": "Comercio y oficios",
    "Técnico electricista": "Comercio y oficios",
}

anonymized["occupation_group"] = anonymized["occupation"].map(occupation_to_group)

anonymized = anonymized.drop(columns=["occupation"])

anonymized.head()

,loyalty_card_number,annual_spend,customer_id,age_group,region,occupation_group
0,********0001,2580000,CUST-EFF698B261D2,20-29,Andina,Comercio y oficios
1,********0002,5400000,CUST-E9F4EC0BD860,30-39,Pacífica,Tecnología y diseño
2,********0003,5320000,CUST-E2C7317583CA,30-39,Andina,Salud y educación
3,********0004,4290000,CUST-6AD4C8020C2B,60-69,Andina,Servicios profesionales
4,********0005,3670000,CUST-92569771F46C,20-29,Andina,Comercio y oficios


In [34]:
# Verificación

auxiliary_attack = auxiliary.copy()

auxiliary_attack["age_group"] = pd.cut(
    auxiliary_attack["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

auxiliary_attack["department"] = auxiliary_attack["city"].map(city_to_department)

auxiliary_attack["region"] = auxiliary_attack["department"].map(department_to_region)

auxiliary_attack["occupation_group"] = auxiliary_attack["occupation"].map(
    occupation_to_group
)

quasi_identifiers = ["age_group", "region", "occupation_group"]

candidates = auxiliary_attack.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

number_of_candidates = (
    candidates.groupby("name").size().reindex(auxiliary["name"], fill_value=0)
)

unique_matches = candidates[candidates["name"].map(number_of_candidates).eq(1)]

print(
    "Perfiles reidentificados de forma única:",
    number_of_candidates.eq(1).sum(),
    "de",
    len(auxiliary),
)

unique_matches.head()

Perfiles reidentificados de forma única: 0 de 20


,name,age,city,occupation,age_group,department,region,occupation_group,loyalty_card_number,annual_spend,customer_id


In [35]:
# Paso 8: se almacena el archivo

anonymized.to_csv("../submission/anonymized.csv", index=False)

In [36]:
# Paso 9: evaluación de la utilidad analítica conservada

utility_comparison = pd.DataFrame(
    {
        "dimension": ["Edad", "Ubicación", "Ocupación"],
        "detalle_original": [
            raw["age"].nunique(),
            raw["city"].nunique(),
            raw["occupation"].nunique(),
        ],
        "detalle_anonimizado": [
            anonymized["age_group"].nunique(),
            anonymized["region"].nunique(),
            anonymized["occupation_group"].nunique(),
        ],
    }
)

utility_comparison

,dimension,detalle_original,detalle_anonimizado
0,Edad,48,5
1,Ubicación,12,3
2,Ocupación,12,4
